<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/Encoder_Transformer_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Architecture Implementation

## 1. InputEmbeddings

In [ ]:
import torch
import math
import torch.nn as nn

class InputEmbeddings(nn.Module):

  def __init__(self, vocab_size : int, d_model : int):
    super().__init__()
    self.d_model = d_model
    self.embeddings = nn.Embedding(vocab_size,d_model)

  def forward(self, x):
    return self.embeddings(x) * math.sqrt(self.d_model)

##2. Positional Encodings

In [ ]:
class PositionalEncodings(nn.Module):

  def __init__(self, d_model : int, seq_len : int, dropout : float = 0.1):
    super().__init_a_()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(seq_len,d_model)
    positions = torch.arange(0,seq_len,dtype=torch.float32).reshape(-1,1)

    div_term = torch.pow(10000,torch.arange(0,d_model,2,dtype=torch.float32) / d_model)

    pe[:,0::2] = torch.sin(positions / div_term)
    pe[:,1::2] = torch.cos(positions / div_term)

    pe = pe.unsqueeze(0)
    self.register_buffer("pe",pe)

  def forward(self,x):
    return self.dropout(x + self.pe[:,:x.shape[1],:])

##3. MultiHead Attention (with KV Cache)

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, d_model : int, num_heads : int, dropout : float = 0.1, use_kv_cache = False):
    super().__init__()

    self.d_model = d_model
    assert (d_model % num_heads == 0), "d_model must be divisible by num_heads"
    self.num_heads = num_heads
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model,d_model)
    self.d_head = d_model // num_heads
    self.dropout = nn.Dropout(dropout)
    self.use_kv_cache = use_kv_cache

  @staticmethod
  def attention(q,k,v, mask, dropout=None):
    d_k = q.shape[-1]
    scaled_dot_product = torch.matmul(q,k.transpose(-1,-2)) / math.sqrt(d_k)

    if mask is not None:
      scaled_dot_product = scaled_dot_product.masked_fill(mask==0,float("-inf"))

    scaled_dot_product = torch.softmax(scaled_dot_product,dim=-1)
    if dropout is not None:
      scaled_dot_product = dropout(scaled_dot_product)

    return torch.matmul(scaled_dot_product,v)


  def forward(self,x,mask, cache_k = None, cache_v = None):

    q = self.W_q(x)
    k = self.W_k(x)
    v = self.W_v(x)
    B,seq_len,_ = q.shape
    # (B,seq_len,d_model) --> (B,seq_len,num_heads,d_head) --> (B,num_heads,seq_len,d_head)
    q = q.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)
    k = k.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)
    v = v.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)

    if self.use_kv_cache:
      if cache_k is not None and cache_v is not None:
        k = torch.cat([cache_k,k],dim=2)
        v = torch.cat([cache_v,v],dim=2)

      new_cache_k = k
      new_cache_v = v
    else:
      new_cache_k = None
      new_cache_v = None

    output = MultiHeadAttention.attention(q,k,v, mask,self.dropout)

    # (B,num_heads,seq_len,d_head) --> (B,seq_len,num_heads,d_head)
    output = output.transpose(1,2).contiguous()
    output = output.view(B,seq_len,-1)

    return self.W_o(output),new_cache_k,new_cache_v

##4. Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, d_model : int, d_ff : int, dropout : float = 0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model,d_ff)
    self.linear2 = nn.Linear(d_ff,d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.linear2(self.dropout(torch.relu(self.linear1(x))))

##5. Residual Connections

In [ ]:
class ResidualConnections(nn.Module):

  def __init__(self, d_model : int, dropout: float = 0.1):
    super().__init__()
    self.layer_norm = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, sublayers):
    return self.layer_norm(x + self.dropout(sublayers(x)))

## General Encoder Block

In [ ]:
class EncoderBlock(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int = 8,dropout : float = 0.1):
    super().__init__()

    self.d_model = d_model
    self.d_ff = d_ff
    self.num_heads = num_heads
    self.dropout = dropout

    self.attention = MultiHeadAttention(d_model,num_heads,dropout,use_kv_cache=False)
    self.residuals = nn.ModuleList([
        ResidualConnections(d_model,dropout) for _ in range(2)
    ])
    self.feed_forward = FeedForward(d_model,d_ff,dropout)

  def forward(self, x, mask):
    attn_sublayer = lambda x : self.attention(x,mask)[0]
    x = self.residuals[0](x, attn_sublayer)
    ff_sublayer = self.feed_forward
    return self.residuals[1](x,ff_sublayer)

## Complete Encoder

In [ ]:
class Encoder(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int = 8, N : int = 6, dropout : float = 0.1):
    super().__init__()
    self.N = N
    self.layers = nn.ModuleList([
        EncoderBlock(d_model, d_ff, num_heads, dropout) for _ in range(N)
    ])
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x, mask):
    for layer in self.layers:
      x = layer(x, mask)

    return self.norm(x)

# Encoder Block For Token Classification (NER)

In [ ]:
class EncoderForTokenClassification(nn.Module):

    def __init__(self, vocab_size: int, seq_len: int, d_model: int, d_ff: int,
                 num_heads: int, N: int, num_labels: int, dropout: float = 0.1):
        super().__init__()

        self.embeddings = InputEmbeddings(vocab_size, d_model)
        self.positional_encodings = PositionalEncodings(d_model, seq_len, dropout)

        self.encoder = `aEncoder(
            d_model=d_model,
            d_ff=d_ff,
            num_heads=num_heads,
            N=N,
            dropout=dropout
        )

        self.classification_head = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, mask):
        x = self.embeddings(input_ids)

        x = self.positional_encodings(x)

        x = self.encoder(x, mask)

        logits = self.classification_head(x)

        return logits

# Encoder For Sentence Classification

In [ ]:
class EncoderForSentenceClassification(nn.Module):

    def __init__(self, vocab_size: int, seq_len: int, d_model: int, d_ff: int,
                 num_heads: int, N: int, num_labels: int, dropout: float = 0.1):
        super().__init__()

        self.embeddings = InputEmbeddings(vocab_size, d_model)
        self.positional_encodings = PositionalEncodings(d_model, seq_len, dropout)

        self.encoder = Encoder(
            d_model=d_model,
            d_ff=d_ff,
            num_heads=num_heads,
            N=N,
            dropout=dropout
        )

        self.classification_head = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, mask):
        x = self.embeddings(input_ids)

        x = self.positional_encodings(x)

        x = self.encoder(x, mask)

        logits = self.classification_head(x[:,0])

        return logits

In [ ]:
!pip install -q transformers datasets

# Training Encoder Model for NER Task (Token Classification)

In [ ]:
from datasets import load_dataset

ds = load_dataset("lawinsider/uk_ner_contracts")
ds

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
tokenizer.special_tokens_map

In [ ]:
ds['train'].features['ner_tags'].feature.names

### Aligning labels with token ids

In [ ]:
def preprocess_fn(examples):

  tokenized_inputs = tokenizer(
      examples["tokens"],
      truncation=True,
      is_split_into_words=True,
      padding="max_length",
      max_length = 512
  )

  labels = []
  for i,label in enumerate(examples['ner_tags']):
    word_ids = tokenized_inputs.word_ids(batch_index=i)

    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
      if word_idx is None:
        label_ids.append(-100)
      elif word_idx != previous_word_idx:
        label_ids.append(label[word_idx])
      else:
        label_ids.append(-100)
      previous_word_idx = word_idx

    labels.append(label_ids)

  tokenized_inputs["labels"] = labels
  return tokenized_inputs

In [ ]:
processed_ds = ds.map(preprocess_fn, batched=True)
processed_ds = processed_ds.remove_columns(['id', 'tokens', 'ner_tags'])

In [ ]:
torch_ds = processed_ds.with_format("torch")
torch_ds

In [ ]:
train_ds = torch_ds['train']
val_ds = torch_ds['validation']
test_ds = torch_ds['test']

len(train_ds),len(val_ds),len(test_ds)

In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=16, shuffle=False)

In [ ]:
len(train_dl),len(val_dl),len(test_dl)

In [ ]:
NUM_LABELS = len(ds['train'].features['ner_tags'].feature.names)

model = EncoderForTokenClassification(
    vocab_size = tokenizer.vocab_size,
    seq_len = 512,
    d_model = 768,
    d_ff = 2048,
    num_heads = 8,
    N = 8,
    num_labels = NUM_LABELS,
    dropout = 0.1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model initialized on {device}")

In [ ]:
train_ds[0]['attention_mask'].shape

In [ ]:
from torch.optim import Adam
from tqdm import tqdm

loss_fn = nn.CrossEntropyLoss()

optimizer = Adam(model.parameters(),lr=5e-5)

num_epochs = 5
num_labels = 5

for epoch in range(num_epochs):

  model.train()
  train_loss = 0.0

  for batch in tqdm(train_dl,desc = f"Epoch {epoch + 1} [Training]"):

    inputs_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device).unsqueeze(1).unsqueeze(2)
    labels = batch['labels'].to(device)

    logits = model(inputs_ids,attention_mask)

    loss = loss_fn(logits.view(-1, num_labels), labels.view(-1))

    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(train_dl)
  print(f"Epoch {epoch + 1} [Training] Loss: {train_loss:.3f}")

  model.eval()
  val_loss = 0.0

  with torch.no_grad():
    for batch in tqdm(val_dl, desc=f"Epoch {epoch + 1} [Validation]"):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device).unsqueeze(1).unsqueeze(2)
      labels = batch['labels'].to(device)

      logits = model(input_ids, attention_mask)

      loss = loss_fn(logits.view(-1, num_labels), labels.view(-1))
      val_loss += loss.item()

  val_loss = val_loss / len(val_dl)
  print(f"Epoch {epoch + 1} - Average Validation Loss: {val_loss:.4f}")

print("Training finished!!!!")

In [ ]:
!pip install -q evaluate seqeval

In [ ]:
import evaluate
import numpy as np

# Load the official "seqeval" metric
metric = evaluate.load("seqeval")

# Get the label names from your dataset
label_names = ds['train'].features['ner_tags'].feature.names

model.eval() # Put model in evaluation mode
all_predictions = []
all_labels = []

# We don't need gradients for evaluation
with torch.no_grad():
    for batch in tqdm(test_dl, desc="Evaluating on Test Set"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Expand the mask to 4D for the model
        expanded_mask = attention_mask.unsqueeze(1).unsqueeze(2)

        logits = model(input_ids, expanded_mask)

        # Convert logits to predictions (shape: B, S)
        predictions = torch.argmax(logits, dim=-1)

        # --- Crucial Part: Filter out -100 labels ---
        # We need to compare label strings, not IDs

        # Loop through each sentence in the batch
        for i in range(labels.shape[0]):
            true_labels = []
            pred_labels = []

            for j in range(labels.shape[1]):
                if labels[i, j] != -100: # This is a real label
                    true_labels.append(label_names[labels[i, j]])
                    pred_labels.append(label_names[predictions[i, j]])

            all_labels.append(true_labels)
            all_predictions.append(pred_labels)

# Let seqeval compute the metrics
results = metric.compute(predictions=all_predictions, references=all_labels)
print(results)

# Training Encoder Model For Pair-wise Sentence Classification

In [ ]:
ds = load_dataset("nvidia/Aegis-AI-Content-Safety-Dataset-2.0")
ds

In [ ]:
ds['train'][0]

In [ ]:
unique_labels = ds['train'].unique('prompt_label')

print(unique_labels)

## Encoder with Dual Classification Heads for Paired Text

In [ ]:
class EncoderForPairsClassification(nn.Module):

    def __init__(self, vocab_size: int, seq_len: int, d_model: int, d_ff: int,
                 num_heads: int, N: int, num_prompt_labels: int, num_response_labels : int,dropout: float = 0.1):
        super().__init__()

        self.embeddings = InputEmbeddings(vocab_size, d_model)
        self.positional_encodings = PositionalEncodings(d_model, seq_len, dropout)

        self.encoder = Encoder(
            d_model=d_model,
            d_ff=d_ff,
            num_heads=num_heads,
            N=N,
            dropout=dropout
        )

        self.prompt_head = nn.Linear(d_model, num_prompt_labels)
        self.response_head = nn.Linear(d_model, num_response_labels)

    def forward(self, input_ids, mask):
        x = self.embeddings(input_ids)

        x = self.positional_encodings(x)

        x = self.encoder(x, mask)
        cls_output = x[:,0]
        prompt_logits = self.prompt_head(cls_output)
        response_logits = self.response_head(cls_output)

        return prompt_logits,response_logits

In [ ]:
prompt_label_names = ds['train'].unique('prompt_label')
response_label_names = ds['train'].unique('response_label')

prompt_label2id = {name: i for i, name in enumerate(prompt_label_names)}
response_label2id = {name: i for i, name in enumerate(response_label_names)}

num_prompt_labels = len(prompt_label_names)
num_response_labels = len(response_label_names)

print(f"Prompt labels: {num_prompt_labels}, Response labels: {num_response_labels}")

In [ ]:
print(prompt_label2id)

In [ ]:
print(response_label2id)

In [ ]:
def preprocess_fn(examples):

    # Ensure empty responses are handled properly
    responses = [r if r is not None else "" for r in examples["response"]]

    tokenized_output = tokenizer(
        examples["prompt"],
        responses,
        truncation=True,
        padding="max_length",
        max_length=768
    )

    # Create the two label columns
    tokenized_output["prompt_label_id"] = [prompt_label2id[label] for label in examples["prompt_label"]]
    tokenized_output["response_label_id"] = [response_label2id[label] for label in examples["response_label"]]

    return tokenized_output

In [ ]:
column_names_to_remove = ds['train'].column_names

processed_ds = ds.map(
    preprocess_fn,
    batched=True,
    remove_columns=column_names_to_remove
)

print(processed_ds)

In [ ]:
torch_ds = processed_ds.with_format("torch")
torch_ds

In [ ]:
train_ds = torch_ds['train']
val_ds = torch_ds['validation']
test_ds = torch_ds['test']

len(train_ds),len(val_ds),len(test_ds)

In [ ]:
train_dl = DataLoader(train_ds,batch_size=16,shuffle=True)
val_dl = DataLoader(val_ds,batch_size=16,shuffle=False)
test_dl = DataLoader(test_ds,batch_size=16,shuffle=False)

In [ ]:
model = EncoderForPairsClassification(
    vocab_size = tokenizer.vocab_size,
    seq_len = 768,
    d_model = 768,
    d_ff = 2048,
    num_heads = 8,
    N = 8,
    num_prompt_labels = num_prompt_labels,
    num_response_labels = num_response_labels,
    dropout = 0.1)

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model initialized on {device}")

In [ ]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

loss_fn = CrossEntropyLoss(label_smoothing=0.1)
optimizer = Adam(model.parameters(), lr=5e-5)

NUM_EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Starting Training")

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for idx, batch in enumerate(tqdm(train_dl, desc=f"Epoch {epoch+1} [Training]")):

        input_ids = batch['input_ids'].to(device)

        attention_mask = batch['attention_mask'].to(device).unsqueeze(1).unsqueeze(2)
        prompt_labels = batch['prompt_label_id'].to(device)
        response_labels = batch['response_label_id'].to(device)

        prompt_logits, response_logits = model(input_ids, attention_mask)

        prompt_loss = loss_fn(prompt_logits, prompt_labels)
        response_loss = loss_fn(response_logits, response_labels)

        total_loss = prompt_loss + response_loss

        train_loss += total_loss.item()

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_dl)
    print(f"Epoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_dl, desc=f"Epoch {epoch + 1} [Validation]"):
            input_ids = batch['input_ids'].to(device)

            attention_mask = batch['attention_mask'].to(device).unsqueeze(1).unsqueeze(2)
            prompt_labels = batch['prompt_label_id'].to(device)
            response_labels = batch['response_label_id'].to(device)

            prompt_logits, response_logits = model(input_ids, attention_mask)

            loss_prompt = loss_fn(prompt_logits, prompt_labels)
            loss_response = loss_fn(response_logits, response_labels)

            total_loss = loss_prompt + loss_response
            val_loss += total_loss.item()

    avg_val_loss = val_loss / len(val_dl)
    print(f"Epoch {epoch + 1} - Average Validation Loss: {avg_val_loss:.4f}")

print("Training finished!!!!")